# **TASK 1: ADVANCED WEB SCRAPING — Real Estate Data**

**Step 1: Install & Import Libraries**

In [2]:
# Installing the tools we need
!pip install requests beautifulsoup4 pandas

# Importing them
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

print("All good! Libraries are ready.")

All good! Libraries are ready.


**Step 2: Scrape the Data**

In [6]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

In [7]:
# Added Ahmedabad to the list
cities = ["mumbai", "new-delhi", "bangalore", "ahmedabad"]

all_properties = []

for city in cities:
    print(f"Scraping {city}...")

    url = f"https://www.magicbricks.com/property-for-sale/residential-real-estate?cityName={city}&page=1"
    response = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, "lxml")

    cards = soup.find_all("div", class_="mb-srp__card")
    print(f"  Found {len(cards)} listings")

    for card in cards:
        title_tag = card.find("h2", class_="mb-srp__card--title")
        title = title_tag.text.strip() if title_tag else "N/A"
        location = title_tag["title"] if title_tag and title_tag.has_attr("title") else "N/A"

        price_tag = card.find("div", class_="mb-srp__card__price--amount")
        price = price_tag.text.strip() if price_tag else "N/A"

        area_tag = card.find("div", {"data-summary": "carpet-area"})
        area = area_tag.find("div", class_="mb-srp__card__summary--value").text.strip() if area_tag else "N/A"

        status_tag = card.find("div", {"data-summary": "status"})
        status = status_tag.find("div", class_="mb-srp__card__summary--value").text.strip() if status_tag else "N/A"

        all_properties.append({
            "City": city.capitalize(),
            "Title": title,
            "Location": location,
            "Price": price,
            "Area": area,
            "Status": status
        })

    time.sleep(2)

df = pd.DataFrame(all_properties)
print(f"\nTotal properties: {len(df)}")
print(df["City"].value_counts())  # Shows count per city

Scraping mumbai...
  Found 30 listings
Scraping new-delhi...
  Found 30 listings
Scraping bangalore...
  Found 30 listings
Scraping ahmedabad...
  Found 30 listings

Total properties: 120
City
Mumbai       30
New-delhi    30
Bangalore    30
Ahmedabad    30
Name: count, dtype: int64


**Step 3: Preview the Data**

In [8]:
# Convert our list into a table
df = pd.DataFrame(all_properties)

# See the first 10 rows
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
df.head(10)

Total rows: 120
Total columns: 6


,City,Title,Location,Price,Area,Status
0,Mumbai,3 BHK Flat for Sale in Mamtora Shree Vallabh...,3 BHK Flat for Sale in Mamtora Shree Vallabh...,₹2.63 Cr,849 sqft,Ready to Move
1,Mumbai,"2 BHK Flat for Sale in Kalpataru Crest, Kalp...","2 BHK Flat for Sale in Kalpataru Crest, Kalp...",₹1.95 Cr,778 sqft,Ready to Move
2,Mumbai,"3 BHK Flat for Sale in Adani The Views, Adan...","3 BHK Flat for Sale in Adani The Views, Adan...",₹3.79 Cr,1022 sqft,Ready to Move
3,Mumbai,"1 BHK Flat for Sale in Sethia Marine View, S...","1 BHK Flat for Sale in Sethia Marine View, S...",₹86.6 Lac,385 sqft,Ready to Move
4,Mumbai,"3 BHK Flat for Sale in Mahatama Gandhi Road,...","3 BHK Flat for Sale in Mahatama Gandhi Road,...",₹2.16 Cr,720 sqft,Ready to Move
5,Mumbai,"4 BHK Flat for Sale in Kalpataru Amare, Kalp...","4 BHK Flat for Sale in Kalpataru Amare, Kalp...",₹13.73 Cr,1899 sqft,Poss. by Dec '27
6,Mumbai,"1 BHK Flat for Sale in Shraddha Gold Crest, ...","1 BHK Flat for Sale in Shraddha Gold Crest, ...",₹1.20 Cr,406 sqft,Poss. by Dec '26
7,Mumbai,"3 BHK Flat for Sale in Godrej Sky Terraces, ...","3 BHK Flat for Sale in Godrej Sky Terraces, ...",₹7.60 Cr,1516 sqft,Poss. by Mar '29
8,Mumbai,Office Space for Sale in Kandivali East Mumbai,Office Space for Sale in Kandivali East Mumbai,₹1.85 Cr,654 sqft,Ready to Move
9,Mumbai,3 BHK Flat for Sale in Lodha New Cuffe Parad...,3 BHK Flat for Sale in Lodha New Cuffe Parad...,₹3.70 Cr,952 sqft,Poss. by Dec '28


**Step 4 (Fix): Find the Right Location Tag**

In [9]:
# Let's peek inside one card's HTML to find the correct location tag
response = requests.get(
    "https://www.magicbricks.com/property-for-sale/residential-real-estate?cityName=mumbai&page=1",
    headers=headers
)
soup = BeautifulSoup(response.text, "lxml")

# Grab just the first card
first_card = soup.find("div", class_="mb-srp__card")

# Print its full HTML so we can see what's inside
print(first_card)

<div class="mb-srp__card"><div class="mb-srp__card__container"><div class="mb-srp__card__photo has-video"><div class="mb-srp__card__photo__fig"><div class="mb-srp__card__photo__fig"><span class="mb-srp__card__photo__fig--count">11<span class="sign-plus">+</span> Photos</span><div class="srpcard-video show-shimmer"><video autoplay="" class="srpCardVideoPlayer" height="auto" loop="" muted="" playsinline="" poster="" preload="none" style="display:block" width="100%"></video></div><div class="mb-srp__card__photo__fig--post">Posted:  <!-- --> <!-- -->Yesterday</div></div></div><div class="mb-srp__card__ads"><div class="mb-srp__card__ads__shield"><div class="mb-srp__card__ads__shield--item" data-shieldicon="agent-certified">Certified Agent</div></div><div class="mb-srp__card__ads__info"><div class="mb-srp__card__ads__info--letter">A</div><div class="mb-srp__card__ads__info--right"><div class="mb-srp__card__ads__info--name">Arihant Associates</div><div class="mb-srp__card__ads__info--served">

**Step 5: Fix & Re-Scrape with Correct Tags**

In [10]:
all_properties = []

for city in cities:
    print(f"Scraping {city}...")

    url = f"https://www.magicbricks.com/property-for-sale/residential-real-estate?cityName={city}&page=1"
    response = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, "lxml")

    cards = soup.find_all("div", class_="mb-srp__card")
    print(f"  Found {len(cards)} listings")

    for card in cards:
        # Title
        title_tag = card.find("h2", class_="mb-srp__card--title")
        title = title_tag.text.strip() if title_tag else "N/A"

        # Location is inside the title attribute of h2
        location = title_tag["title"] if title_tag and title_tag.has_attr("title") else "N/A"

        # Price
        price_tag = card.find("div", class_="mb-srp__card__price--amount")
        price = price_tag.text.strip() if price_tag else "N/A"

        # Area
        area_tag = card.find("div", {"data-summary": "carpet-area"})
        area = area_tag.find("div", class_="mb-srp__card__summary--value").text.strip() if area_tag else "N/A"

        # Status
        status_tag = card.find("div", {"data-summary": "status"})
        status = status_tag.find("div", class_="mb-srp__card__summary--value").text.strip() if status_tag else "N/A"

        all_properties.append({
            "City": city.capitalize(),
            "Title": title,
            "Location": location,
            "Price": price,
            "Area": area,
            "Status": status
        })

    time.sleep(2)

# Preview
df = pd.DataFrame(all_properties)
print(f"\nTotal properties: {len(df)}")
df.head(5)

Scraping mumbai...
  Found 30 listings
Scraping new-delhi...
  Found 30 listings
Scraping bangalore...
  Found 30 listings
Scraping ahmedabad...
  Found 30 listings

Total properties: 120


,City,Title,Location,Price,Area,Status
0,Mumbai,3 BHK Flat for Sale in Mamtora Shree Vallabh...,3 BHK Flat for Sale in Mamtora Shree Vallabh...,₹2.63 Cr,849 sqft,Ready to Move
1,Mumbai,"2 BHK Flat for Sale in Kalpataru Crest, Kalp...","2 BHK Flat for Sale in Kalpataru Crest, Kalp...",₹1.95 Cr,778 sqft,Ready to Move
2,Mumbai,"3 BHK Flat for Sale in Adani The Views, Adan...","3 BHK Flat for Sale in Adani The Views, Adan...",₹3.79 Cr,1022 sqft,Ready to Move
3,Mumbai,"1 BHK Flat for Sale in Sethia Marine View, S...","1 BHK Flat for Sale in Sethia Marine View, S...",₹86.6 Lac,385 sqft,Ready to Move
4,Mumbai,"3 BHK Flat for Sale in Mahatama Gandhi Road,...","3 BHK Flat for Sale in Mahatama Gandhi Road,...",₹2.16 Cr,720 sqft,Ready to Move


**Step 6: Save the Data as a CSV File**

In [11]:
# Save to a CSV file
df.to_csv("real_estate_data.csv", index=False)

print(f"Saved! Total properties: {len(df)}")
print(f"Columns: {list(df.columns)}")

# Download it to your computer
from google.colab import files
files.download("real_estate_data.csv")

Saved! Total properties: 120
Columns: ['City', 'Title', 'Location', 'Price', 'Area', 'Status']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>